In [ ]:
# Hail mary

import numpy as np
''''''
# File containing calibration data, make sure to have run through compressor.py
#calibration_data_dir = r'C:\Users\burri\Documents\PET\TimeCalibration\CalibrationDataPreparer\compressed_photopeak_cut_10th.bin'
calibration_data_dir = r'C:\Users\burri\Documents\PET\TimeCalibration\CalibrationDataPreparer\compressed_photopeak_cut.bin'

# File containing geometric offsets table, can generate from map with geotabler.py
geo_offsets_dir = r'C:\Users\burri\Documents\PET\TimeCalibration\CalibrationDataPreparer\geometrictimeoffsets1.bin'

# Directory to output calibration tsvs to
output_dir = f'C:/Users/burri/Documents/PET/TimeCalibration/CalibrationDataPreparer/'

chunksize = 128 # Chunk size to calibrate in, 128 is stable, adjust according to RAM, must be a factor of 3072

''''''

import numpy as np
import os
from tqdm import tqdm
#import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd


# Reading from photopeak cut data
num_rows = os.path.getsize(calibration_data_dir) // 6
data = np.memmap(calibration_data_dir, dtype = np.int16, mode = 'r', shape=(num_rows, 3))

# Event counts
eventcounts = np.zeros((3072 * 3072), dtype = np.int32)
for i in tqdm(range(0, num_rows, 1000000), desc='Event counts'):
    chunk = data[i:i+1000000]
    ids = np.asarray(chunk[:, [0, 1]], dtype = np.int32)
    indices = ids[:, 0] * 3072 + ids[:, 1]
    np.add.at(eventcounts, indices, 1)
eventcounts = eventcounts.reshape((3072, 3072))

# For applying geometric offset
geooffsets = np.fromfile(geo_offsets_dir, dtype = np.float32).reshape((3072, 3072))

# For fitting peaks, gaussian + background
def gaussianbg(x, a, mu, c, bg):
    return a * np.exp(-(x - mu)**2 / (2 * c**2)) + bg

# For histogramming
bins = np.linspace(-2500, 2500, 200) # Consistent histogram bins
bincenters = [(bins[n] + bins[n + 1]) / 2 for n in range(len(bins) - 1)]

lookup = np.zeros((3072, 3072), dtype = np.int32)

for chunknum in tqdm(range(3072 // chunksize), desc = f'Big loop'):
    chunkidls = range(chunknum * chunksize, (chunknum + 1) * chunksize)
    backadj = chunknum * chunksize

    chunkchannelevents = [np.zeros((np.sum(eventcounts[idl]), 2), dtype = np.int16) for idl in chunkidls]
    writeheads = np.zeros(chunksize, dtype=np.int32)

    # Collect events for idls in chunkidls
    for i in tqdm(range(0, num_rows, 1000000), desc = 'Read chunk', leave = False):
        for idl in chunkidls:
            chunk = data[i:i+1000000] 
            chunk = chunk[chunk[:,0] == idl]
            chunk = chunk[:, [1, 2]]
            chunkchannelevents[idl - backadj][writeheads[idl - backadj]:writeheads[idl - backadj] + len(chunk)] = chunk
            writeheads[idl - backadj] += len(chunk)
    break

Big loop:   0%|          | 0/24 [00:07<?, ?it/s]


In [8]:
print(chunkchannelevents[0])

[[ 1204. -1174.]
 [  483. -2764.]
 [  962.   400.]
 ...
 [   45. -9083.]
 [ 1299.  5257.]
 [  976. -1306.]]


In [18]:
for idl in tqdm(chunkidls, desc = 'Geo offsets', leave = False):
    for idr in range(3072):
        timediffs = chunkchannelevents[0][chunkchannelevents[0][:,0] == 0]
        break
    break        

In [ ]:
calibration_data_dir = r'C:\Users\burri\Documents\PET\TimeCalibration\CalibrationDataPreparer\compressed_photopeak_cut.bin'

# Reading from photopeak cut data
num_rows = os.path.getsize(calibration_data_dir) // 6
data = np.memmap(calibration_data_dir, dtype = np.int16, mode = 'r', shape=(num_rows, 3))

# Event counts
eventcounts = np.zeros((3072 * 3072), dtype = np.int32)
for i in tqdm(range(0, num_rows, 1000000), desc='Event counts'):
    chunk = data[i:i+1000000]
    ids = np.asarray(chunk[:, [0, 1]], dtype = np.int32)
    indices = ids[:, 0] * 3072 + ids[:, 1]
    np.add.at(eventcounts, indices, 1)
eventcounts = eventcounts.reshape((3072, 3072))


Event counts: 100%|██████████| 196/196 [00:13<00:00, 14.73it/s]


In [ ]:
print(np.max(eventcounts))

127


20.683556026882595
20.683556026882595


In [28]:
geo1 = np.fromfile(r'C:\Users\burri\Documents\PET\TimeCalibration\CalibrationDataPreparer\geometrictimeoffsets1.bin', dtype = np.float32).reshape((3072, 3072))
print(geo1[0:5][0:5])
geo2 = np.fromfile(r'C:\Users\burri\Documents\PET\TimeCalibration\CalibrationDataPreparer\geometrictimeoffsets2.bin', dtype = np.float32).reshape((3072, 3072))
print(geo2[0:5][0:5])

[[-172.8254    -183.47687   -194.35161   ...  -20.933342   -16.671286
   -29.40884  ]
 [-168.98862   -179.6974    -190.63058   ...  -16.743858   -12.501929
   -25.199928 ]
 [-165.18185   -175.95273   -186.80083   ...  -12.481989    -8.289249
   -20.86309  ]
 [-172.83301   -183.48499   -194.30841   ...  -20.890606   -16.640266
   -29.345589 ]
 [-161.21553   -172.04001   -183.04294   ...   -8.353551    -4.1593246
   -16.756538 ]]
[[172.8254    183.47687   194.35161   ...  20.933342   16.671286
   29.40884  ]
 [168.98862   179.6974    190.63058   ...  16.743858   12.501929
   25.199928 ]
 [165.18185   175.95273   186.80083   ...  12.481989    8.289249
   20.86309  ]
 [172.83301   183.48499   194.30841   ...  20.890606   16.640266
   29.345589 ]
 [161.21553   172.04001   183.04294   ...   8.353551    4.1593246
   16.756538 ]]
